# Networks to test

In [ ]:
import torch
device = "cuda"

In [ ]:
# Test forward pass with a dummy input
def test_forward_pass(model, dimensions=(1, 1, 96, 96, 96)):
    print("\n--- Testing Forward Pass ---")
    
    # Create a dummy input tensor with the specified dimensions
    dummy_input = torch.randn(*dimensions).to(device)  # e.g., (batch_size, in_channels, D, H, W)

    # Perform the forward pass (within a no_grad context for efficiency during testing)
    with torch.no_grad():
        try:
            output = model(dummy_input)
            print("Forward pass successful!")
            print(f"Output shape: {output.shape}")
            expected_shape = (dimensions[0], 1, dimensions[2], dimensions[3], dimensions[4])
            assert output.shape == expected_shape, f"Expected output shape {expected_shape}, but got {output.shape}"
        except Exception as e:
            print(f"Error during forward pass: {e}")


## Swin UNETR
* Data availability: Transformer-based models like Swin UNETR can perform effectively even with limited labeled data, making them suitable for scenarios with smaller datasets.

In [ ]:
import torch
from monai.networks.nets import SwinUNETR
from os.path import join

def fix_checkpoint_keys(state_dict):
    """
    This function replaces the 'module.' with 'swinViT.' 
    and the linear layers '.mlp.fc' with '.mlp.linear'
    """
    fixed_state_dict = {}
    for key, value in state_dict.items():
        new_key = key
        if key.startswith("module."):
            new_key = key.replace('module.', 'swinViT.')  # Replace module by swinViT
            new_key = new_key.replace('.mlp.fc', '.mlp.linear')  # Replace module by swinViT
             
        fixed_state_dict[new_key] = value
    return fixed_state_dict

def load_pretrained_swinvit(ckpt_path, img_size=(96, 96, 96), in_channels=1, out_channels=14, feature_size=48, use_checkpoint=True, verbose=False):
    """
    Load a pretrained SwinViT model from a checkpoint.

    Args:
        ckpt_path (str): Path to the checkpoint file.
        img_size (tuple): Image size for the model.
        in_channels (int): Number of input channels.
        out_channels (int): Number of output channels.
        feature_size (int): Feature size for the model.
        use_checkpoint (bool): Whether to use checkpointing.

    Returns:
        model: The loaded SwinUNETR model.
    """

    # Load the checkpoint
    model_dict = torch.load(ckpt_path, weights_only=False)
    state_dict = model_dict.get("state_dict", model_dict) 

    # Fix keys in the checkpoint
    state_dict = fix_checkpoint_keys(state_dict)

    # Initialize the model
    model = SwinUNETR(
        img_size=img_size,
        in_channels=in_channels,
        out_channels=out_channels,
        feature_size=feature_size,
        use_checkpoint=use_checkpoint,
    )

    if verbose:
        # Check for mismatches
        model_keys = set(model.state_dict().keys())
        checkpoint_keys = set(state_dict.keys())

        # Check matching and mismatching keys
        loaded_keys = model_keys & checkpoint_keys
        not_loaded_keys = model_keys - checkpoint_keys
        extra_keys_in_checkpoint = checkpoint_keys - model_keys

        print(f"\n✅ Loaded {len(loaded_keys)} keys:")
        for k in sorted(loaded_keys):
            print(f"  {k}")

        print(f"\n❌ Not Loaded ({len(not_loaded_keys)}):")
        for k in sorted(not_loaded_keys):
            print(f"  {k}")

        print(f"\n📦 Extra keys in checkpoint (ignored): {len(extra_keys_in_checkpoint)}")
        for k in sorted(extra_keys_in_checkpoint):
            print(f"  {k}")

    # Load the state_dict into the model
    model.load_state_dict(state_dict, strict=False)
    print("✅ Model weights loaded (non-strict mode).")
    return model


ckpt_path = join("/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/Synthrad2025_by_Faking_it/Synthrad2025_by_Faking_it/Synthrad2025_by_Faking_it/Synthrad2025_by_Faking_it/Synthrad2025_by_Faking_it/Synthrad2025_by_Faking_it/Synthrad2025_by_Faking_it/Synthrad2025_by_Faking_it/src/Synthetic-CT-generation-from-MRI-using-3D-transformer-based-denoising-diffusion-model/network/pre_trained/SwinUNETR/model_swinvit.pt")
model = load_pretrained_swinvit(ckpt_path, verbose=True, )

In [ ]:
# Load the model trained on the BTCV multi-organ dataset
from os.path import join

def load_pretrained_SwinUNETR(ckpt_path, img_size=(96, 96, 96), in_channels=1, out_channels=14, feature_size=48, use_checkpoint=True, verbose=False):
    """
    Load a pretrained SwinUNETR model from a checkpoint.

    Args:
        ckpt_path (str): Path to the checkpoint file.
        img_size (tuple): Image size for the model.
        in_channels (int): Number of input channels.
        out_channels (int): Number of output channels.
        feature_size (int): Feature size for the model.
        use_checkpoint (bool): Whether to use checkpointing.

    Returns:
        model: The loaded SwinUNETR model.
    """
    # Load the checkpoint
    model_dict = torch.load(ckpt_path, weights_only=False)

    # Initialize the model
    model = SwinUNETR(
        img_size=img_size,
        in_channels=in_channels,
        out_channels=out_channels,
        feature_size=feature_size,
        use_checkpoint=use_checkpoint,
    )

    # Extract state_dict from the checkpoint
    state_dict = model_dict.get("state_dict", model_dict)
    if verbose:
        # Check for mismatches
        model_keys = set(model.state_dict().keys())
        checkpoint_keys = set(state_dict.keys())

        # Check matching and mismatching keys
        loaded_keys = model_keys & checkpoint_keys
        not_loaded_keys = model_keys - checkpoint_keys
        extra_keys_in_checkpoint = checkpoint_keys - model_keys

        print(f"\n✅ Loaded {len(loaded_keys)} keys:")
        for k in sorted(loaded_keys):
            print(f"  {k}")

        print(f"\n❌ Not Loaded ({len(not_loaded_keys)}):")
        for k in sorted(not_loaded_keys):
            print(f"  {k}")

        print(f"\n📦 Extra keys in checkpoint (ignored): {len(extra_keys_in_checkpoint)}")
        for k in sorted(extra_keys_in_checkpoint):
            print(f"  {k}")

    # Load the state_dict into the model
    model.load_state_dict(state_dict)
    print("✅ Model weights loaded (strict mode).")
    return model


ckpt_path = join('/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/src/Synthetic-CT-generation-from-MRI-using-3D-transformer-based-denoising-diffusion-model/network/pre_trained/SwinUNETR/swin_unetr.base_5000ep_f48_lr2e-4_pretrained.pt')
model = load_pretrained_SwinUNETR(ckpt_path, verbose=True)

## Basic U-Net

In [ ]:
from monai.networks.nets import UNet
net = UNet(
    spatial_dims=3,
    in_channels=2,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

test_forward_pass(model, dimensions=(1, 2, 96, 96, 96))

### Total Segmentator based nnUNet

In [ ]:
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
import nnunetv2
from nnunetv2.utilities.find_class_by_name import recursive_find_python_class
from batchgenerators.utilities.file_and_folder_operations import load_json, join, isfile, maybe_mkdir_p, isdir, subdirs, \
    save_json
from nnunetv2.utilities.plans_handling.plans_handler import PlansManager, ConfigurationManager
from nnunetv2.utilities.label_handling.label_handling import determine_num_input_channels



def load_pretrained_TotalSegmentator(ckpt_path, verbose):
    """
    Load a pretrained TotalSegmentator model from a checkpoint.

    Args:
        ckpt_path (str): Path to the checkpoint directory.
        verbose (bool): Whether to print detailed information about the loading process.

    Returns:
        model: The loaded TotalSegmentator model.
    """
    checkpoint_name = 'checkpoint_final.pth'

    # Load dataset and plans JSON files
    dataset_json = load_json(join(ckpt_path, 'dataset.json'))
    plans = load_json(join(ckpt_path, 'plans.json'))
    plans_manager = PlansManager(plans)

    # Define folds to use
    use_folds = [0] # Always use fold 0
    if isinstance(use_folds, str):
        use_folds = [use_folds]

    # Load network weights from checkpoints
    parameters = []
    for i, f in enumerate(use_folds):
        f = int(f) if f != 'all' else f
        checkpoint = torch.load(
            join(ckpt_path, f'fold_{f}', checkpoint_name),
            map_location=torch.device('cpu'),
            weights_only=False
        )
        if i == 0:
            trainer_name = checkpoint['trainer_name']
            configuration_name = checkpoint['init_args']['configuration']
            inference_allowed_mirroring_axes = checkpoint.get('inference_allowed_mirroring_axes', None)

        parameters.append(checkpoint['network_weights'])

    # Get configuration manager and input channels
    configuration_manager = plans_manager.get_configuration(configuration_name)
    num_input_channels = determine_num_input_channels(plans_manager, configuration_manager, dataset_json)

    # Find the trainer class
    trainer_class = recursive_find_python_class(
        join(nnunetv2.__path__[0], "training", "nnUNetTrainer"),
        trainer_name,
        'nnunetv2.training.nnUNetTrainer'
    )

    # Build the network architecture
    model = trainer_class.build_network_architecture(
        configuration_manager.network_arch_class_name,
        configuration_manager.network_arch_init_kwargs,
        configuration_manager.network_arch_init_kwargs_req_import,
        num_input_channels,
        plans_manager.get_label_manager(dataset_json).num_segmentation_heads,
        enable_deep_supervision=False
    )

    # Verbose output for key mismatches
    if verbose:
        model_keys = set(model.state_dict().keys())
        checkpoint_keys = set(parameters[0].keys())

        loaded_keys = model_keys & checkpoint_keys
        not_loaded_keys = model_keys - checkpoint_keys
        extra_keys_in_checkpoint = checkpoint_keys - model_keys

        print(f"\n✅ Loaded {len(loaded_keys)} keys:")
        for k in sorted(loaded_keys):
            print(f"  {k}")

        print(f"\n❌ Not Loaded ({len(not_loaded_keys)}):")
        for k in sorted(not_loaded_keys):
            print(f"  {k}")

        print(f"\n📦 Extra keys in checkpoint (ignored): {len(extra_keys_in_checkpoint)}")
        for k in sorted(extra_keys_in_checkpoint):
            print(f"  {k}")

    # Load the state_dict into the model
    model.load_state_dict(parameters[0])
    print("✅ Model weights loaded (strict mode).")
    return model

ckpt_path = '/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/src/Synthetic-CT-generation-from-MRI-using-3D-transformer-based-denoising-diffusion-model/network/pre_trained/TotalSegmentator/Dataset297_TotalSegmentator_total_3mm_1559subj/nnUNetTrainer_4000epochs_NoMirroring__nnUNetPlans__3d_fullres'
model = load_pretrained_TotalSegmentator(ckpt_path, verbose=False)
    

# Apply new losses



In [ ]:
import os
import torch
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
import nnunetv2
from nnunetv2.utilities.find_class_by_name import recursive_find_python_class
from batchgenerators.utilities.file_and_folder_operations import load_json, join
from nnunetv2.utilities.plans_handling.plans_handler import PlansManager
from nnunetv2.utilities.label_handling.label_handling import determine_num_input_channels
def load_pretrained_TotalSegmentator(ckpt_path):
    """
    Load a pretrained TotalSegmentator model from a checkpoint.

    Args:
        ckpt_path (str): Path to the checkpoint directory.
        verbose (bool): Whether to print detailed information about the loading process.

    Returns:
        model: The loaded TotalSegmentator model.
    """
    os.environ['nnUNet_raw'] = ''
    os.environ['nnUNet_preprocessed'] = ''
    os.environ['nnUNet_results'] = ''
    checkpoint_name = 'checkpoint_final.pth'
    try:
        ckpt_path = ckpt_path.split('/fold_0')[0]
    except:
        raise Exception('The checkpoint_final.pth needs to be inside of the folder fold_0')
    dataset_json = load_json(join(ckpt_path, 'dataset.json'))
    plans = load_json(join(ckpt_path, 'plans.json'))
    plans_manager = PlansManager(plans)
    use_folds = [0]
    if isinstance(use_folds, str):
        use_folds = [use_folds]
    parameters = []
    for i, f in enumerate(use_folds):
        f = int(f) if f != 'all' else f
        checkpoint = torch.load(join(ckpt_path, f'fold_{f}', checkpoint_name), map_location=torch.device('cpu'), weights_only=False)
        if i == 0:
            trainer_name = checkpoint['trainer_name']
            configuration_name = checkpoint['init_args']['configuration']
            inference_allowed_mirroring_axes = checkpoint.get('inference_allowed_mirroring_axes', None)
        parameters.append(checkpoint['network_weights'])

    configuration_manager = plans_manager.get_configuration(configuration_name)
    num_input_channels = determine_num_input_channels(plans_manager, configuration_manager, dataset_json)
    trainer_class = recursive_find_python_class(join(nnunetv2.__path__[0], 'training', 'nnUNetTrainer'), trainer_name, 'nnunetv2.training.nnUNetTrainer')
    model = trainer_class.build_network_architecture(configuration_manager.network_arch_class_name, configuration_manager.network_arch_init_kwargs, configuration_manager.network_arch_init_kwargs_req_import, num_input_channels, plans_manager.get_label_manager(dataset_json).num_segmentation_heads, enable_deep_supervision=False)
    
    model.load_state_dict(parameters[0])

    print('✅ Model weights loaded (strict mode).')
    return (model)
seg_model = load_pretrained_TotalSegmentator(ckpt_path="../../metrics/evaluation/.totalsegmentator/nnunet/results/Dataset297_TotalSegmentator_total_3mm_1559subj/nnUNetTrainer_4000epochs_NoMirroring__nnUNetPlans__3d_fullres/fold_0/checkpoint_final.pth")



In [ ]:
from monai.transforms import (
    LoadImaged,
    CropForegroundd,
    RandSpatialCropSamplesd,
    AsDiscreted,
    EnsureTyped,
    EnsureType,
    ScaleIntensityRanged,
    ResampleToMatchd,
    EnsureChannelFirstd,
    GridPatchd,
    CopyItemsd,
    CenterSpatialCropd,
    ResizeWithPadOrCropd,
    SpatialCropd,
    Resize,
    Orientationd,
    ResizeWithPadOrCrop

)
from monai.data import Dataset, DataLoader
import numpy as np
import nibabel as nib
transforms = [
    LoadImaged(keys=["ct", "mask", "mri", "gt_seg"]),
    EnsureChannelFirstd(keys=["ct", "mask", "mri", "gt_seg"]),
    Orientationd(keys=["ct", "mask", "mri"], axcodes='RAS'),
    CropForegroundd(keys=["ct", "mask", "mri"], source_key="mask", allow_smaller=False),  # Crop based on mask
    CopyItemsd(keys=["ct", "mri", "mask", "gt_seg"], times=1, names=["ct_fullres", "mri_fullres", "mask_fullres", "gt_seg_fullres"], allow_missing_keys=False),
    ResizeWithPadOrCropd(keys=["ct_fullres", "ct", "mri_fullres", "mask_fullres", "gt_seg"], spatial_size=(336, 336, 128), mode=["minimum", "minimum", "minimum", "minimum", "minimum"]), # To ensure the input size is 336, 336, 128
    SpatialCropd(keys=["ct", "mask", "mri", "gt_seg"], roi_size=(128, 128, 32), roi_center=(100, 100, 100)), # Now we know that's in the center. Later we need to define a random center selection to save the it and paste back
    #CenterSpatialCropd(keys=["ct", "mask", "mri"], roi_size=(128, 128, 32)), # Now we know that's in the center. Later we need to define a random center selection to save the it and paste back
]

### With spacing from 1x1x3 to 3x3x3 we need input shape of 336, 336, 128 to match the total segmentator 3mm spacing

file_paths = [
    {
        "ct": "/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/Synthrad2025_by_Faking_it/Dataset/synthRAD2025_Task1_Train/Task1/AB/1ABA009/ct.mha",
        "mask": "/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/Synthrad2025_by_Faking_it/Dataset/synthRAD2025_Task1_Train/Task1/AB/1ABA009/mask.mha",
        "mri": "/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/Synthrad2025_by_Faking_it/Dataset/synthRAD2025_Task1_Train/Task1/AB/1ABA009/mr.mha",
        "gt_seg": "/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/Synthrad2025_by_Faking_it/trash/gt_seg.mha"
    }
]
# Load dataset
dataset = Dataset(data=file_paths, transform=transforms)
dataloader = DataLoader(dataset, batch_size=1)  # Process one patient at a time

In [ ]:
import sys
sys.path.append("/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/src/metrics/functions")
sys.path.append("/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/src/metrics/evaluation")
sys.path.append("/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/Synthrad2025_by_Faking_it/src/Synthetic-CT-generation-from-MRI-using-3D-transformer-based-denoising-diffusion-model")
sys.path.append("/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/src/Synthetic-CT-generation-from-MRI-using-3D-transformer-based-denoising-diffusion-model")
sys.path.append("/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/src/Synthetic-CT-generation-from-MRI-using-3D-transformer-based-denoising-diffusion-model/network")

from network.Diffusion_model_transformer import *
num_channels=64
attention_resolutions="32,16,8"
channel_mult = (1, 2, 3, 4)
num_heads=[4,4,8,16]
window_size = [[4,4,4],[4,4,4],[4,4,2],[4,4,2]]
num_res_blocks = [2,2,2,2]
sample_kernel=([2,2,2],[2,2,1],[2,2,1],[2,2,1]),
use_scale_shift_norm = True
resblock_updown = False
dropout = 0
use_checkpoint=False

attention_ds = []
for res in attention_resolutions.split(","):
    attention_ds.append(int(res))

A_to_B_model = SwinVITModel(
        image_size=(128, 128, 32),
        in_channels=2,
        model_channels=num_channels,
        out_channels=2,
        dims=3,
        sample_kernel = sample_kernel,
        num_res_blocks=num_res_blocks,
        attention_resolutions=tuple(attention_ds),
        dropout=dropout,
        channel_mult=channel_mult,
        num_classes=None,
        use_checkpoint=use_checkpoint,
        use_fp16=False,
        num_heads=num_heads,
        window_size = window_size,
        num_head_channels=64,
        num_heads_upsample=-1,
        use_scale_shift_norm=use_scale_shift_norm,
        resblock_updown=resblock_updown,
        use_new_attention_order=False,
    )


In [ ]:
resize_to_seg = Resize(
    spatial_size=(112, 112, 128), 
    size_mode='all', 
    mode='trilinear', 
    align_corners=None, 
    anti_aliasing=False, 
    anti_aliasing_sigma=None,
    lazy=False
    )

resize_back_seg = Resize(
    spatial_size=(336, 336, 128), 
    size_mode='all', 
    mode='nearest', 
    align_corners=None, 
    anti_aliasing=False, 
    anti_aliasing_sigma=None,
    lazy=False
    )

AB = [
        2, # kidney right
        3, # kidney left
        5, # liver
        6, # stomach
        *range(10, 14+1), #lungs
        *range(26, 50+1), #vertebrae
        51, #heart
        79, # spinal cord
        *range(92, 115+1), # ribs
        116 #sternum
    ]


def normalize_for_totalseg(image: torch.Tensor) -> torch.Tensor:
    """
    Normalizes a CT image tensor using predefined mean, std, and intensity bounds.
    """
    mean_intensity = -50.38697721419439
    std_intensity = 503.39235619144
    lower_bound = -1004.0
    upper_bound = 1588.0

    image = torch.clamp(image, lower_bound, upper_bound)
    print(f"image min: {torch.min(image)}, image max: {torch.max(image)}")
    image = image - mean_intensity
    image = image / torch.max(torch.tensor(std_intensity, device=image.device), torch.tensor(1e-8, device=image.device))

    return image

In [ ]:
affine = np.array([[1., 0., 0., 0.],
    [0., 1., 0., 0.],
    [0., 0., 3., 0.],
    [0., 0., 0., 1.]])


In [ ]:
def predict_seg(fullres_w_synthpatch):
    """
    Performs segmentation on a batch of 3D medical images.

    Args:
        fullres_w_synthpatch (torch.Tensor): Input tensor of shape (B, C, D, H, W) e.g., (2, 1, 128, 128, 128),
            where B is the batch size, and C, D, H, W are channel and spatial dimensions.

    Returns:
        None. (Assumes further processing or storage of results happens externally.)
    """
    batch_preds = []
    batch_logits = []
    for i in range(fullres_w_synthpatch.shape[0]):
        single_image = fullres_w_synthpatch[i]  # Shape: (C, D, H, W)

        resized = resize_to_seg(single_image)

        nifti_img = nib.Nifti1Image(resized[0].numpy(), affine=np.eye(4)) # REMOVE
        nib.save(nifti_img, "/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/Synthrad2025_by_Faking_it/trash/resized.nii.gz") # REMOVE

        normed = normalize_for_totalseg(resized).squeeze(0).T

        # Predict segmentation
        pred = seg_model(normed.unsqueeze(0).unsqueeze(0)).squeeze()  # Squeeze all in one go
        print(f"pred: {pred.shape}")
        batch_logits.append(pred)
        seg_pred = torch.softmax(pred, dim=0).argmax(0).T
        batch_preds.append(seg_pred.unsqueeze(0))
        

    return torch.stack(batch_preds), torch.stack(batch_logits)



In [ ]:
import torch
import random

def random_foreground_crop(
    ct_fullres: torch.Tensor, 
    mri_fullres: torch.Tensor, 
    mask: torch.Tensor, 
    crop_size=(128, 128, 32)
):
    """
    Randomly crop the same patch from ct_fullres and mri_fullres (shape: B,C,H,W,D),
    ensuring the patch contains foreground voxels from mask.

    Args:
        ct_fullres (torch.Tensor): CT volume, shape (1, 1, 336, 336, 128)
        mri_fullres (torch.Tensor): MRI volume, same shape as ct_fullres
        mask (torch.Tensor): Binary mask tensor, same shape as ct_fullres
        crop_size (tuple): Desired crop size (crop_H, crop_W, crop_D), default (128, 128, 32)

    Returns:
        cropped_ct (torch.Tensor): Cropped CT volume patch (1, 1, crop_H, crop_W, crop_D)
        cropped_mri (torch.Tensor): Cropped MRI volume patch (1, 1, crop_H, crop_W, crop_D)
        cropped_mask (torch.Tensor): Cropped mask patch (1, 1, crop_H, crop_W, crop_D)
        crop_coords (tuple): (start_y, start_x, start_z, end_y, end_x, end_z)
    """
    B, C, H, W, D = ct_fullres.shape
    assert (B, C, H, W, D) == (1, 1, 336, 336, 128), "Input tensors must have shape (1,1,336,336,128)"
    assert mri_fullres.shape == ct_fullres.shape, "mri_fullres must have the same shape as ct_fullres"
    assert mask.shape == ct_fullres.shape, "mask must have the same shape as ct_fullres"

    crop_H, crop_W, crop_D = crop_size

    mask_squeezed = mask[0, 0]

    fg_indices = (mask_squeezed == 1).nonzero(as_tuple=False)

    if fg_indices.size(0) == 0:
        raise ValueError("Mask contains no foreground voxels.")

    idx = random.randint(0, fg_indices.size(0) - 1)
    center_y, center_x, center_z = fg_indices[idx]

    start_y = torch.clamp(center_y - crop_H // 2, min=0, max=H - crop_H)
    start_x = torch.clamp(center_x - crop_W // 2, min=0, max=W - crop_W)
    start_z = torch.clamp(center_z - crop_D // 2, min=0, max=D - crop_D)

    end_y = start_y + crop_H
    end_x = start_x + crop_W
    end_z = start_z + crop_D

    cropped_ct = ct_fullres[:, :, start_y:end_y, start_x:end_x, start_z:end_z]
    cropped_mri = mri_fullres[:, :, start_y:end_y, start_x:end_x, start_z:end_z]
    cropped_mask = mask[:, :, start_y:end_y, start_x:end_x, start_z:end_z]

    crop_coords = (start_y.item(), start_x.item(), start_z.item(), end_y.item(), end_x.item(), end_z.item())

    return cropped_ct, cropped_mri, cropped_mask, crop_coords


In [ ]:
from monai.metrics import DiceMetric
from monai.losses import DiceLoss

compute_dice_loss = DiceLoss(
    include_background=True, 
    to_onehot_y=True, 
    sigmoid=False, 
    softmax=True, 
    other_act=None, 
    squared_pred=False, 
    jaccard=False, 
    reduction="mean", 
    smooth_nr=1e-05, 
    smooth_dr=1e-05, 
    batch=False, 
    weight=None, 
    )

compute_dice = DiceMetric(
    include_background=True, 
    reduction="mean", 
    get_not_nans=False,
    ignore_empty=True,
    num_classes=None, 
    return_with_label=False)

In [ ]:
for batch in dataloader:
    ct_fullres =  batch['ct_fullres']
    mri_fullres = batch['mri_fullres']
    mask = batch['mask_fullres']
    gt_seg_full_res = batch['gt_seg_fullres']

    cropped_ct, cropped_mri, cropped_mask, place = random_foreground_crop(
        ct_fullres=ct_fullres, 
        mri_fullres=mri_fullres, 
        mask=mask, 
        crop_size=(128, 128, 32)
    )
    start_y, start_x, start_z, end_y, end_x, end_z = place

    nifti_img = nib.Nifti1Image(cropped_ct[0][0].numpy(), affine=np.eye(4))
    nib.save(nifti_img, "/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/Synthrad2025_by_Faking_it/trash/cropped_ct.nii.gz")

    nifti_img = nib.Nifti1Image(cropped_mri[0][0].numpy(), affine=np.eye(4))
    nib.save(nifti_img, "/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/Synthrad2025_by_Faking_it/trash/cropped_mri.nii.gz")

    nifti_img = nib.Nifti1Image(cropped_mask[0][0].numpy(), affine=np.eye(4))
    nib.save(nifti_img, "/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/Synthrad2025_by_Faking_it/trash/cropped_mask.nii.gz")

    fullres_w_synthpatch = ct_fullres.clone()
    fullres_w_synthpatch[:, :, start_y:end_y, start_x:end_x, start_z:end_z] = cropped_ct

    nifti_img = nib.Nifti1Image(ct_fullres[0][0].numpy(), affine=np.eye(4))
    nib.save(nifti_img, "/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/Synthrad2025_by_Faking_it/trash/ct_fullres.nii.gz")

    nifti_img = nib.Nifti1Image(fullres_w_synthpatch[0][0].numpy(), affine=np.eye(4))
    nib.save(nifti_img, "/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/Synthrad2025_by_Faking_it/trash/fullres_w_synthpatch.nii.gz")

    nifti_img = nib.Nifti1Image(gt_seg_full_res[0][0].numpy(), affine=np.eye(4))
    nib.save(nifti_img, "/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/Synthrad2025_by_Faking_it/trash/gt_seg_full_res.nii.gz")


    if torch.equal(fullres_w_synthpatch, batch['ct_fullres']):
        print("The pasted back image is equal to the original fullres image.")
    else:
        print("The pasted back image is NOT equal to the original fullres image.")


    # Perform the predication segmentation
    seg_model.eval()
    for param in seg_model.parameters(): 
        param.requires_grad = False
    seg_pred, logits = predict_seg(fullres_w_synthpatch)
    print(f"seg_pred: {seg_pred.shape}")

    seg_pred = F.interpolate(seg_pred.float() , size=(336, 336, 128), mode='nearest')
    print(f"seg_pred: {seg_pred.shape}")

    # Save a sample to NIfTI if needed
    nifti_img = nib.Nifti1Image(seg_pred[0][0].detach().cpu().numpy().astype(np.int16), affine=np.eye(4))
    nib.save(nifti_img, f"/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/Synthrad2025_by_Faking_it/trash/seg.nii.gz")
    
    nifti_img = nib.Nifti1Image(fullres_w_synthpatch[0][0].numpy(), affine=np.eye(4))
    nib.save(nifti_img, "/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/Synthrad2025_by_Faking_it/trash/fullres_w_synthpatch.nii.gz")
    
    print(f"DSC={compute_dice(seg_pred, gt_seg_full_res)}")
    
    gt_seg_full_res = F.interpolate(gt_seg_full_res , size=(112, 112, 128), mode='nearest') 
    print(f"DSC_loss={compute_dice_loss(logits.permute(0,1,4,3,2),gt_seg_full_res)}")
    

In [ ]:
import torch
import SimpleITK as sitk
import numpy as np

# -------------------------------
# 1. Load reference image (for metadata)
ref_path = "/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/Synthrad2025_by_Faking_it/Dataset/synthRAD2025_Task1_Train/Task1/AB/1ABA005/mask.mha"
ref_img = sitk.ReadImage(ref_path)

# -------------------------------
# 2. Convert your tensor to numpy
# Assumes seg_pred shape is (1, 1, D, H, W)
seg_np = seg_pred.squeeze().cpu().numpy().astype(np.uint8)  # shape: (D, H, W)

# -------------------------------
# 3. Create a SimpleITK image
pred_img = sitk.GetImageFromArray(seg_np.T)

# -------------------------------
# 4. Copy metadata from reference
pred_img.SetSpacing(ref_img.GetSpacing())
pred_img.SetOrigin(ref_img.GetOrigin())
pred_img.SetDirection(ref_img.GetDirection())

# -------------------------------
# 5. Save to .mha
out_path = "/media/andreferreira/a8176754-c206-4f75-8aba-7d08aed3f47a/AI_work/SynthRad2025/Synthrad2025_by_Faking_it/trash/gt_seg.mha"  # Change path as needed
sitk.WriteImage(pred_img, out_path)
